In [2]:
from ollama import chat
LLM_MODEL = "qwen2.5-coder:3b"
response = chat(
        model=LLM_MODEL,
        messages=[
            {
                "role": "user",
                "content": "hi"
            }
        ]
    )

response

ChatResponse(model='qwen2.5-coder:3b', created_at='2026-09-09T05:27:26.812718157Z', done=True, done_reason='stop', total_duration=9053169195, load_duration=6663377003, prompt_eval_count=30, prompt_eval_duration=910432364, eval_count=22, eval_duration=1402673832, message=Message(role='assistant', content="Hello! How can I assist you today? Is there anything specific you'd like to know or discuss?", thinking=None, images=None, tool_name=None, tool_calls=None), logprobs=None)

## Imports and configurations

In [ ]:
import torch
import sounddevice as sd
import numpy as np
from scipy.io.wavfile import write
from pathlib import Path
import json
import re
import subprocess
import time
from urllib import request
from transformers import AutoTokenizer
import onnxruntime as ort
from ollama import chat
from kokoro import KPipeline
from IPython.display import Audio, display
import soundfile as sf
from queue import Queue, Empty
import threading
# ============================================================
# Configuration
# ============================================================

SAMPLE_RATE = 16000
CHUNK_SIZE = 512

SPEECH_THRESHOLD = 0.5
SILENCE_DURATION = 1.0
MAX_UTTERANCE_SECONDS = 30

LLM_MODEL = "qwen2.5-coder:3b"

WHISPER_DIR = Path("../whisper.cpp")
WHISPER_MODEL = WHISPER_DIR / "models" / "ggml-base.en.bin"
WHISPER_SERVER = WHISPER_DIR / "build" / "bin" / "whisper-server"
WHISPER_SERVER_URL = "http://127.0.0.1:8080"

SHOULD_RESPOND_CLASS = 1
                    chunk, overflowed = stream.read(CHUNK_SIZE)
                    chunk_is_echo = (
                        assistant_speaking_event.is_set()
                        and echo_reference.is_echo(chunk)
                    )
VOICE = "af_heart"
SYSTEM_PROMPT = """You are a concise proactive assistant.
                    is_speech = speech_probability >= SPEECH_THRESHOLD and not chunk_is_echo
Return plain conversational text only: no markdown, bullets, labels, or line breaks.
Return at most two short sentences. Make each sentence 8 to 16 words when possible.
Always finish each sentence with '.', '?' or '!'.
Use the user's language when you understand it; otherwise answer briefly in English."""


### Loading models

In [2]:

# ============================================================
# Load Should AI Respond model
# ============================================================

model, utils = torch.hub.load(
    repo_or_dir="snakers4/silero-vad",
    model="silero_vad",
    trust_repo=True
)

tokenizer = AutoTokenizer.from_pretrained(
    "./should_ai_respond_model"
)

print("Loaded tokenizer")


int8_session = ort.InferenceSession(
    "./should_ai_respond_int8.onnx",
    providers=["CPUExecutionProvider"]
)

print("Loaded INT8 BERT classification model")

pipeline = KPipeline(lang_code="a")
audio_streamer = sd.OutputStream(
            samplerate=24000,
            channels=1,
            dtype="float32",
            blocksize=2048
        )

Using cache found in /home/keerthivardhan/.cache/torch/hub/snakers4_silero-vad_master
/home/keerthivardhan/Desktop/ProductionProjects/autonomus-assistent/.venv/lib/python3.12/site-packages/torch/jit/_serialization.py:176: FutureWarning: `torch.jit.load` is deprecated. Please switch to `torch.export`.
  warnings.warn(


Loaded tokenizer
Loaded INT8 BERT classification model


/home/keerthivardhan/Desktop/ProductionProjects/autonomus-assistent/.venv/lib/python3.12/site-packages/torch/nn/modules/rnn.py:1011: UserWarning: dropout option adds dropout after all but last recurrent layer, so non-zero dropout expects num_layers greater than 1, but got dropout=0.2 and num_layers=1
  super().__init__("LSTM", *args, **kwargs)
/home/keerthivardhan/Desktop/ProductionProjects/autonomus-assistent/.venv/lib/python3.12/site-packages/torch/nn/utils/weight_norm.py:145: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)
/home/keerthivardhan/Desktop/ProductionProjects/autonomus-assistent/.venv/lib/python3.12/site-packages/torch/jit/_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


In [41]:
### Helper
# ============================================================
# Helper
# ============================================================

from collections import deque
from queue import Empty
from scipy.signal import resample_poly


def softmax(x):
    exp_x = np.exp(
        x - np.max(x, axis=1, keepdims=True)
    )
    return exp_x / exp_x.sum(
        axis=1,
        keepdims=True
    )


def clear_queue(q):
    while True:
        try:
            q.get_nowait()
        except Empty:
            break


class EchoReference:
    def __init__(self, output_sample_rate, input_sample_rate, max_seconds=2):
        self.output_sample_rate = output_sample_rate
        self.input_sample_rate = input_sample_rate
        self.samples = deque(maxlen=input_sample_rate * max_seconds)
        self.lock = threading.Lock()

    def add(self, audio):
        audio = np.asarray(audio, dtype=np.float32).reshape(-1)
        if self.output_sample_rate != self.input_sample_rate:
            audio = resample_poly(
                audio,
                self.input_sample_rate,
                self.output_sample_rate
            ).astype(np.float32)
        with self.lock:
            self.samples.extend(audio.tolist())

    def is_echo(self, microphone_chunk, correlation_threshold=0.65):
        microphone_chunk = np.asarray(microphone_chunk, dtype=np.float32).reshape(-1)
        microphone_energy = np.linalg.norm(microphone_chunk)
        if microphone_energy < 1e-4:
            return False

        with self.lock:
            reference = np.asarray(self.samples, dtype=np.float32)

        if reference.size < microphone_chunk.size:
            return False

        microphone_centered = microphone_chunk - microphone_chunk.mean()
        microphone_norm = np.linalg.norm(microphone_centered)
        if microphone_norm < 1e-4:
            return False

        search_start = max(0, reference.size - microphone_chunk.size - 2560)
        search_end = reference.size - microphone_chunk.size
        for start in range(search_start, search_end + 1, 160):
            reference_chunk = reference[start:start + microphone_chunk.size]
            reference_centered = reference_chunk - reference_chunk.mean()
            reference_norm = np.linalg.norm(reference_centered)
            if reference_norm < 1e-4:
                continue
            correlation = np.dot(microphone_centered, reference_centered) / (
                microphone_norm * reference_norm
            )
            if correlation >= correlation_threshold:
                return True

        return False


def start_whisper_server():
    server = subprocess.Popen(
        [
            str(WHISPER_SERVER),
            "-m", str(WHISPER_MODEL),
            "-nt",
            "--host", "127.0.0.1",
            "--port", "8080"
        ],
        stdout=subprocess.DEVNULL,
        stderr=subprocess.STDOUT
    )

    health_url = f"{WHISPER_SERVER_URL}/"
    deadline = time.time() + 30

    while time.time() < deadline:
        try:
            with request.urlopen(health_url, timeout=1):
                return server
        except Exception:
            if server.poll() is not None:
                raise RuntimeError("Whisper server stopped during startup")
            time.sleep(0.1)

    server.terminate()
    raise TimeoutError("Whisper server did not become ready")


def transcribe_with_whisper_server(audio_file, whisper_url=WHISPER_SERVER_URL):
    boundary = f"----assistant-{time.time_ns()}"
    audio_data = Path(audio_file).read_bytes()
    body = b"".join([
        f"--{boundary}\r\n".encode(),
        b'Content-Disposition: form-data; name="file"; filename="utterance.wav"\r\n',
        b"Content-Type: audio/wav\r\n\r\n",
        audio_data,
        f"\r\n--{boundary}\r\n".encode(),
        b'Content-Disposition: form-data; name="response_format"\r\n\r\n',
        b"json\r\n",
        f"--{boundary}--\r\n".encode()
    ])

    http_request = request.Request(
        f"{whisper_url}/inference",
        data=body,
        headers={
            "Content-Type": f"multipart/form-data; boundary={boundary}"
        },
        method="POST"
    )

    with request.urlopen(http_request, timeout=60) as response:
        result = json.loads(response.read().decode("utf-8"))

    return result.get("text", "").strip()


### Listener Worker (VAD + ASR)

In [42]:
def listen_for_speech(audio_queue, stop_event, listen_event, assistant_speaking_event, interrupt_event, echo_reference):
    try:
        with sd.InputStream(
            samplerate=SAMPLE_RATE,
            channels=1,
            dtype="float32",
            blocksize=CHUNK_SIZE
        ) as stream:
            while not stop_event.is_set():
                audio_chunks = []
                speech_started = False
                silence_start = None
                speech_start_time = time.time()
                model.reset_states()

                while not stop_event.is_set():
                    chunk, overflowed = stream.read(CHUNK_SIZE)
                    chunk = chunk[:, 0]
                    chunk_is_echo = (
                        assistant_speaking_event.is_set()
                        and echo_reference.is_echo(chunk)
                    )
                    chunk_tensor = torch.from_numpy(chunk)
                    speech_probability = model(chunk_tensor, SAMPLE_RATE).item()
                    is_speech = speech_probability >= SPEECH_THRESHOLD and not chunk_is_echo

                    if is_speech:
                        if not speech_started:
                            speech_started = True
                            if assistant_speaking_event.is_set():
                                print("Barge-in detected; interrupting assistant.")
                                interrupt_event.set()
                                listen_event.set()
                            else:
                                print("Speech detected...")
                        audio_chunks.append(chunk.copy())
                        silence_start = None

                    elif speech_started:
                        audio_chunks.append(chunk.copy())
                        if silence_start is None:
                            silence_start = time.time()
                        if time.time() - silence_start >= SILENCE_DURATION:
                            print("User finished speaking.")
                            break

                    if speech_started and time.time() - speech_start_time >= MAX_UTTERANCE_SECONDS:
                        print("Maximum speech duration reached.")
                        break

                if stop_event.is_set() or not speech_started:
                    continue

                listen_event.clear()
                speech_finished_at = time.perf_counter()
                audio = np.concatenate(audio_chunks)
                audio_file = Path(f"utterance_{time.time_ns()}.wav")
                write(audio_file, SAMPLE_RATE, audio)
                audio_queue.put((audio_file, speech_finished_at))

    finally:
        print("VAD worker stopped")

In [5]:
def Listener(Listener_BERT_shared_queue, Listen_for_speech_VAD_listener_shared_queue, stop_event, whisper_url, listen_event):

    while not stop_event.is_set():
        try:
            audio_item = Listen_for_speech_VAD_listener_shared_queue.get(timeout=0.5)
        except Empty:
            continue

        if audio_item is None:
            break

        audio_file, speech_finished_at = audio_item
        print("Calling Whisper server...")
        asr_started_at = time.perf_counter()

        try:
            user_text = transcribe_with_whisper_server(audio_file, whisper_url)
        except Exception as error:
            print(f"Whisper failed: {error}")
            listen_event.set()
            continue

        print(f"Whisper latency: {time.perf_counter() - asr_started_at:.3f}s")

        if not user_text:
            print("Whisper returned empty text.")
            listen_event.set()
            continue

        print("\nUser:")
        print(user_text)
        Listener_BERT_shared_queue.put((user_text, speech_finished_at))

    print("Listener: came out of while loop")


### BERT


In [6]:
def ShouldAIRespond(Listener_BERT_shared_queue, tokenizer, int8_session, softmax, LLM_BERT_shared_queue, stop_event, listen_event):

    while not stop_event.is_set():
        try:
            user_item = Listener_BERT_shared_queue.get(timeout=0.5)
        except Empty:
            continue

        if user_item is None:
            print("ShouldAIRespond stopped")
            break

        user_text, speech_finished_at = user_item
        inputs = tokenizer(user_text, return_tensors="np", truncation=True)
        onnx_inputs = {
            "input_ids": inputs["input_ids"],
            "attention_mask": inputs["attention_mask"]
        }
        logits = int8_session.run(None, onnx_inputs)[0]
        probs = softmax(logits)
        prediction = np.argmax(probs, axis=1)[0]
        respond_probability = probs[0][SHOULD_RESPOND_CLASS]
        print(f"Should respond probability: {respond_probability:.3f}")

        if prediction != SHOULD_RESPOND_CLASS:
            print("BERT decided: DON'T RESPOND")
            listen_event.set()
            continue

        print("BERT decided: RESPOND")
        print(f"Speech-to-BERT latency: {time.perf_counter() - speech_finished_at:.3f}s")
        LLM_BERT_shared_queue.put((user_text, speech_finished_at))

    print("BERT: Came out of loop")


### LLM

In [43]:
def split_complete_sentences(buffer):
    sentences = []
    sentence_end = re.compile(r".*?[.!?](?:[\"')\]]+)?(?=\s|$)")

    while True:
        match = sentence_end.match(buffer)
        if match is None:
            break

        sentence = match.group(0).strip()
        if sentence:
            sentences.append(sentence)
        buffer = buffer[match.end():].lstrip()

    return sentences, buffer


def LLM(LLM_MODEL, LLM_BERT_shared_queue, LLM_Kokoro_shared_queue, stop_event, interrupt_event):
    while not stop_event.is_set():
        try:
            user_item = LLM_BERT_shared_queue.get(timeout=0.5)
        except Empty:
            continue

        if user_item is None:
            print("LLM stopped")
            break

        user_text, speech_finished_at = user_item
        stream = chat(
            model=LLM_MODEL,
            messages=[
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": user_text}
            ],
            options={"temperature": 0.2},
            stream=True
        )

        buffer = ""
        first_text_chunk = True

        for chunk in stream:
            if interrupt_event.is_set() or stop_event.is_set():
                break

            token = chunk["message"]["content"]
            print(token, end="", flush=True)
            buffer += token

            sentences, buffer = split_complete_sentences(buffer)
            for sentence in sentences:
                if interrupt_event.is_set():
                    break
                LLM_Kokoro_shared_queue.put((sentence, speech_finished_at, first_text_chunk))
                if first_text_chunk:
                    print(f"\nTime to first TTS text: {time.perf_counter() - speech_finished_at:.3f}s")
                    first_text_chunk = False

        if interrupt_event.is_set() or stop_event.is_set():
            continue

        remaining = buffer.strip()
        if remaining:
            if remaining[-1] not in ".?!":
                remaining += "."
            LLM_Kokoro_shared_queue.put((remaining, speech_finished_at, first_text_chunk))
            if first_text_chunk:
                print(f"\nTime to first TTS text: {time.perf_counter() - speech_finished_at:.3f}s")

        LLM_Kokoro_shared_queue.put(("response_done", None))

    print("LLM: came out of loop")


In [44]:
from queue import Empty

def audio_generator(audio_queue, text_queue, stop_event, assistant_speaking_event, interrupt_event, echo_reference):
    while not stop_event.is_set():
        try:
            text_item = text_queue.get(timeout=0.5)
        except Empty:
            continue

        if text_item is None:
            break

        if text_item[0] == "response_done":
            audio_queue.put(("response_done", None))
            continue

        assistant_speaking_event.set()
        text, speech_finished_at, first_text_chunk = text_item
        generator = pipeline(text, voice=VOICE)

        first_audio_chunk = True
        for _, _, audio in generator:
            if stop_event.is_set() or interrupt_event.is_set():
                break
            if first_audio_chunk and first_text_chunk:
                print(f"Time to first audio generated: {time.perf_counter() - speech_finished_at:.3f}s")
                first_audio_chunk = False
            echo_reference.add(audio)
            audio_queue.put(audio)

        if interrupt_event.is_set():
            clear_queue(text_queue)
            assistant_speaking_event.clear()

    assistant_speaking_event.clear()
    print("audio_generator stopped")

In [45]:
from queue import Empty

def Assistant(audio_queue, audio_streamer, stop_event, listen_event, assistant_speaking_event, interrupt_event):
    audio_streamer.start()

    try:
        while not stop_event.is_set():
            try:
                audio = audio_queue.get(timeout=0.5)
            except Empty:
                continue

            if audio is None:
                break

            if interrupt_event.is_set():
                clear_queue(audio_queue)
                assistant_speaking_event.clear()
                listen_event.set()
                try:
                    audio_streamer.abort()
                except Exception:
                    pass
                continue

            if isinstance(audio, tuple) and audio[0] == "response_done":
                assistant_speaking_event.clear()
                listen_event.set()
                continue

            audio_streamer.write(audio)

    finally:
        print("Stopping stream:", id(audio_streamer))
        audio_streamer.stop()
        audio_streamer.close()
        print("Assistant stopped")

In [46]:
from multiprocessing import Queue as MPQueue, Process, Event as MPEvent

def main():

    whisper_server = start_whisper_server()
    Listen_for_speech_VAD_listener_shared_queue = MPQueue()
    Listener_BERT_shared_queue = MPQueue()
    LLM_BERT_shared_queue = Queue()
    LLM_Kokoro_shared_queue = Queue()
    audio_queue = Queue()

    stop_event = MPEvent()
    listen_event = MPEvent()
    listen_event.set()
    assistant_speaking_event = MPEvent()
    interrupt_event = MPEvent()
    echo_reference = EchoReference(24000, SAMPLE_RATE)

    VAD_worker = threading.Thread(
        target=listen_for_speech,
        args=(Listen_for_speech_VAD_listener_shared_queue, stop_event, listen_event, assistant_speaking_event, interrupt_event, echo_reference),
        name="VAD_worker",
        daemon=True
    )

    Listener_worker = Process(
        target=Listener,
        args=(Listener_BERT_shared_queue, Listen_for_speech_VAD_listener_shared_queue, stop_event, WHISPER_SERVER_URL, listen_event),
        name="Listener"
    )

    ShouldAIRespond_worker = threading.Thread(
        target=ShouldAIRespond,
        args=(Listener_BERT_shared_queue, tokenizer, int8_session, softmax, LLM_BERT_shared_queue, stop_event, listen_event),
        name="ShouldAIRespond_worker",
        daemon=True
    )

    LLM_worker = threading.Thread(
        target=LLM,
        args=(LLM_MODEL, LLM_BERT_shared_queue, LLM_Kokoro_shared_queue, stop_event, interrupt_event),
        name="LLM_worker",
        daemon=True
    )

    Audio_generator_worker = threading.Thread(
        target=audio_generator,
        args=(audio_queue, LLM_Kokoro_shared_queue, stop_event, assistant_speaking_event, interrupt_event, echo_reference),
        name="Audio_generator_worker",
        daemon=True
    )

    Assistant_worker = threading.Thread(
        target=Assistant,
        args=(audio_queue, audio_streamer, stop_event, listen_event, assistant_speaking_event, interrupt_event),
        name="Assistant_worker",
        daemon=True
    )

    for worker in [VAD_worker, Listener_worker, ShouldAIRespond_worker, LLM_worker, Audio_generator_worker, Assistant_worker]:
        worker.start()

    print("All workers have been started.")

    try:
        while True:
            time.sleep(1)

    except KeyboardInterrupt:
        print("\nStopping assistant...")
        stop_event.set()
        listen_event.set()
        interrupt_event.set()
        Listen_for_speech_VAD_listener_shared_queue.put(None)
        Listener_BERT_shared_queue.put(None)
        LLM_BERT_shared_queue.put(None)
        LLM_Kokoro_shared_queue.put(None)
        audio_queue.put(None)

    for worker in [VAD_worker, ShouldAIRespond_worker, LLM_worker, Audio_generator_worker, Assistant_worker]:
        worker.join(timeout=3)
        print(f"{worker.name}: {'stopped' if not worker.is_alive() else 'still running'}")

    Listener_worker.join(timeout=3)
    if Listener_worker.is_alive():
        print(f"Process {Listener_worker.name}: still running, terminating")
        Listener_worker.terminate()
        Listener_worker.join(timeout=2)
    else:
        print(f"Process {Listener_worker.name}: stopped")

    whisper_server.terminate()
    whisper_server.wait(timeout=5)
    Listen_for_speech_VAD_listener_shared_queue.close()
    Listener_BERT_shared_queue.close()
    print("Assistant stopped.")


In [11]:
from viztracer import VizTracer

tracer = VizTracer()

tracer.start()

main()

tracer.stop()
tracer.save("assistant_trace.json")

All workers have been started.
Speech detected...
Calling Whisper server...
User finished speaking.
Whisper latency: 1.915s

User:
(speaking in foreign language)
Should respond probability: 0.692
BERT decided: RESPOND
Speech-to-BERT latency: 1.991s
I'm sorry, I don't understand what you're saying.
Time to first TTS text: 12.678s
 Can you please repeat?Time to first audio generated: 16.316s
Speech detected...
Calling Whisper server...
User finished speaking.
Whisper latency: 2.175s

User:
Yeah, what is LILM?
Should respond probability: 0.786
BERT decided: RESPOND
Speech-to-BERT latency: 2.247s
LILM stands for "Learning Inference and Learning Machines".
Time to first TTS text: 3.825s
 It's a research project focused on developing AI systems that can learn from data to make predictions or decisions.Time to first audio generated: 10.822s
Speech detected...
Calling Whisper server...
User finished speaking.
Whisper latency: 1.876s

User:
No, I'm saying what is the LLM?
Should respond probabi

Process Listener:
Traceback (most recent call last):
  File "/usr/lib/python3.12/multiprocessing/process.py", line 314, in _bootstrap
    self.run()
  File "/usr/lib/python3.12/multiprocessing/process.py", line 108, in run
    self._target(*self._args, **self._kwargs)
  File "/tmp/ipykernel_399198/3486012214.py", line 5, in Listener
    audio_item = Listen_for_speech_VAD_listener_shared_queue.get(timeout=0.5)
                 ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/multiprocessing/queues.py", line 113, in get
    if not self._poll(timeout):
           ^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/multiprocessing/connection.py", line 257, in poll
    return self._poll(timeout)
           ^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/multiprocessing/connection.py", line 440, in _poll
    r = wait([self], timeout)
        ^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/multiprocessing/connection.py", line 1136, in wait
    ready = sel


Stopping assistant...
Stopping stream: 137198210119680
audio_generator stopped
ShouldAIRespond stopped
BERT: Came out of loop
LLM stopped
LLM: came out of loop
VAD worker stopped
VAD_worker: stopped
ShouldAIRespond_worker: stopped
LLM_worker: stopped
Audio_generator_worker: stopped
Assistant stopped
Assistant_worker: stopped
Process Listener: stopped
Assistant stopped.
Loading finish                                        
Total Entries: 579692                                                           
Use the following command to open the report:
vizviewer /home/keerthivardhan/Desktop/ProductionProjects/autonomus-assistent/labs/assistant_trace.json


In [12]:
11
10
7
7


11

- sometimes, tts is not working 
- it is responding to its own voice , becasue listiner thread is keeps on listining 
    - - implement semophose (a boolean variable) indicating listener to listen or not
- optimize stt process
- avg : it is responding in 10sec